# Phase 3 — the assistant, end to end

Retrieve the law, answer from it, cite the source. This notebook loads the model
trained in Phase 2.5 from your Google Drive and wires it to the retriever.

## How to run it

1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**, and allow Google Drive access when asked
3. It takes about **10 minutes** — there is no training here

## What is new since Phase 2.5

Phase 2.5 showed retrieval works, but the confusion set — the 44 hardest
old-versus-new questions — barely improved. The cause was mechanical: 33 of
those answers name two or more provisions (a section *and* its counterpart, and
for merged families every constituent), while the run supplied a single passage.
Asking for more passages hardly helped, because ranking by similarity does not
preferentially surface a provision's counterpart.

So the bot now looks counterparts up deliberately, in the concordance built in
Phase 1. Measured on the assembled context, the provisions an answer needs are
all present for:

| Context assembly | Confusion set |
|---|---|
| similarity only, 1 passage | 15 / 44 |
| similarity only, 3 passages | 35 / 44 |
| similarity only, 5 passages | 38 / 44 |
| **citations + concordance counterparts** | **43 / 44** |

The last section of this notebook checks whether that turns into better answers,
not just better context.

In [ ]:
# ----------------------------------------------------------------- settings
REPO_URL = "https://github.com/suryanshu-g/legal-llm-bot.git"

# The model trained by the Phase 2.5 notebook. Change only if you moved it.
MODEL_DIR = "/content/drive/MyDrive/legal-llm-bot/flan-t5-small-context"
DRIVE_DIR = "/content/drive/MyDrive/legal-llm-bot"

In [ ]:
%pip install -q -U transformers datasets accelerate sentencepiece sentence-transformers faiss-cpu

> **If Colab shows a "RESTART SESSION" button after the install, click it**, then
> carry on from the next cell.

In [ ]:
import os, json, re, sys, time, textwrap, subprocess
from collections import Counter, defaultdict

import numpy as np
import torch

REPO_DIR = "/content/legal-llm-bot"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("already cloned")
else:
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
                       capture_output=True, text=True)
    print(r.stdout or "", r.stderr or "")
    if r.returncode != 0:
        raise RuntimeError("git clone failed")

DATA = os.path.join(REPO_DIR, "data", "processed")
sys.path.insert(0, os.path.join(REPO_DIR, "scripts"))

from google.colab import drive
drive.mount("/content/drive")

if not os.path.isdir(MODEL_DIR):
    raise FileNotFoundError(
        f"No model at {MODEL_DIR}.\n"
        "Run notebooks/finetune_flan_t5_small_contextaware.ipynb first, or set "
        "MODEL_DIR to wherever the trained model was saved.")
print("model:", MODEL_DIR)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

---

## Loading the bot

`scripts/bot.py` is the whole assistant: it assembles context, builds the prompt
the model was fine-tuned on, generates, and reports which passages the answer
rests on. The same module runs outside Colab, so nothing here is notebook-only.

In [ ]:
from bot import Bot, DISCLAIMER

t0 = time.time()
bot = Bot(model_dir=MODEL_DIR)
print(f"loaded in {time.time() - t0:.1f}s")
print(f"corpus: {len(bot.retriever.meta)} passages | "
      f"concordance: {len(bot.counterparts)} provisions with counterparts")

---

## Ask it things

Eight questions with known answers, covering each capability. Read the answers
and the sources — this is the demo material for the video.

In [ ]:
QUESTIONS = [
    "What does BNS Section 103 cover?",
    "Which BNS section replaced IPC Section 302?",
    "Which CrPC section corresponds to BNSS Section 173?",
    "Is an offence under BNS Section 303 bailable?",
    "Which court tries an offence under BNS Section 103?",
    "An offence was committed on 15 August 2024. Does the IPC or the BNS apply?",
    "Which BNS section corresponds to IPC Section 124A?",
    "Does BNSS Section 482 deal with the same subject as CrPC Section 482?",
]

for q in QUESTIONS:
    a = bot.ask(q)
    print("=" * 78)
    print("Q:", q)
    print("A:", textwrap.fill(a.text, 76, subsequent_indent="   "))
    for c in a.citations:
        print(f"   source: {c['source']}  {c['source_url']}")

### Staying inside its brief

The project forbids the assistant from giving legal advice, suggesting ways to
evade liability, or posing as an advocate. The model was trained to refuse, but a
trained refusal is a tendency rather than a guarantee, so `bot.py` also checks
the question before answering. These should all be refused.

In [ ]:
for q in ["Can you be my lawyer and represent me in court?",
          "How can I avoid being convicted under BNS Section 318?",
          "Tell me a loophole in BNS Section 103.",
          "Should I plead guilty?",
          "What does BNS Section 63 cover?"]:          # this one must NOT refuse
    a = bot.ask(q)
    print(f"[{'REFUSED' if a.refused else 'answered'}] {q}")
    print("   ", textwrap.shorten(a.text, 150, placeholder=" ..."))

---

## Does the counterpart lookup improve the answers?

The confusion set, twice over the same 44 questions: once with a single
similarity-retrieved passage, as Phase 2.5 ran it, and once with the bot's
assembled context. The metric that matters is whether the answer names every
provision the gold answer names.

In [ ]:
ARTICLES = re.compile(r"\b(a|an|the)\b")
PUNCT = re.compile(r"[^\w\s]")


def normalise(s):
    """SQuAD-style normalisation: case, punctuation and articles removed."""
    s = PUNCT.sub(" ", s.lower())
    return " ".join(ARTICLES.sub(" ", s).split())


def exact_match(pred, gold):
    return float(normalise(pred) == normalise(gold))


def token_f1(pred, gold):
    p, g = normalise(pred).split(), normalise(gold).split()
    if not p or not g:
        return float(p == g)
    common = Counter(p) & Counter(g)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision, recall = overlap / len(p), overlap / len(g)
    return 2 * precision * recall / (precision + recall)

In [ ]:
# ---- statutory reference extraction ---------------------------------------
# The dataset writes citations several ways - "BNS Section 103", "BNS 103",
# "Section 477 of the Code of Criminal Procedure, 1973", and enumerations like
# "IPC Sections 415, 417, 418, 419 and 420" - so all of those have to parse.
ACT_ALIASES = {
    "BHARATIYA NYAYA SANHITA": "BNS", "BNS": "BNS",
    "BHARATIYA NAGARIK SURAKSHA SANHITA": "BNSS", "BNSS": "BNSS",
    "BHARATIYA SAKSHYA ADHINIYAM": "BSA", "BSA": "BSA",
    "INDIAN PENAL CODE": "IPC", "IPC": "IPC",
    "CODE OF CRIMINAL PROCEDURE": "CRPC", "CRPC": "CRPC", "CR.P.C": "CRPC",
    "INDIAN EVIDENCE ACT": "IEA", "EVIDENCE ACT": "IEA", "IEA": "IEA",
}
# Longest alias first, so "Indian Evidence Act" wins over "Evidence Act".
_ACTS = "|".join(re.escape(a) for a in sorted(ACT_ALIASES, key=len, reverse=True))
_NUMS = r"\d+[A-Za-z]{0,2}(?:\s*(?:,|and|&)\s*\d+[A-Za-z]{0,2})*"

# The optional \d{4} skips the year in "the Indian Penal Code, 1860" so that a
# year is never mistaken for a section number.
REF_ACT_FIRST = re.compile(
    r"(" + _ACTS + r")\b[ ,]*(?:\d{4}[ ,]*)?(?:Sections?|ss?\.)?\s*(" + _NUMS + r")",
    re.I)
REF_SEC_FIRST = re.compile(
    r"Sections?\s*(" + _NUMS + r")\s*(?:of\s+(?:the\s+)?)(" + _ACTS + r")\b", re.I)


def _numbers(blob):
    for part in re.split(r"\s*(?:,|and|&)\s*", blob):
        m = re.fullmatch(r"(\d+)([A-Z]{0,2})", part.strip().upper())
        if m and int(m.group(1)) < 1000:      # >= 1000 is a year, not a section
            yield m.group(1) + m.group(2)


def extract_refs(text):
    """The set of statutory references a passage cites, e.g. {'BNS 103'}."""
    found = set()
    for pat, act_first in ((REF_ACT_FIRST, True), (REF_SEC_FIRST, False)):
        for m in pat.finditer(text):
            act = (m.group(1) if act_first else m.group(2)).upper().rstrip(".")
            nums = m.group(2) if act_first else m.group(1)
            canon = ACT_ALIASES.get(act)
            if canon:
                found.update(f"{canon} {n}" for n in _numbers(nums))
    return found


# Verify against the forms that actually occur before trusting the metric.
checks = [
    ("IPC Section 302 corresponds to BNS Section 103.", {"IPC 302", "BNS 103"}),
    ("CrPC 438 is now BNSS 482.", {"CRPC 438", "BNSS 482"}),
    ("Section 65B of the Indian Evidence Act, 1872", {"IEA 65B"}),
    ("BNS Section 318 absorbs IPC Sections 415, 417, 418, 419 and 420.",
     {"BNS 318", "IPC 415", "IPC 417", "IPC 418", "IPC 419", "IPC 420"}),
    ("The Bharatiya Nyaya Sanhita, 2023 replaced the Indian Penal Code, 1860.",
     set()),
]
for text, want in checks:
    got = extract_refs(text)
    assert got == want, f"{text!r}\n  got  {sorted(got)}\n  want {sorted(want)}"
print(f"reference extractor: {len(checks)}/{len(checks)} checks pass")

# Why this metric earns its place: token F1 hardly notices a wrong section.
gold = "IPC Section 302 corresponds to BNS Section 103 (Punishment for murder)."
wrong = "IPC Section 302 corresponds to BNS Section 302 (Punishment for murder)."
print(f"  a wrong-section answer scores token F1 {token_f1(wrong, gold):.3f} "
      f"but fails the citation check ({extract_refs(gold) <= extract_refs(wrong)})")

In [ ]:
def score(rows, preds, metas=None):
    """Per-row metrics, plus the aggregate and the per-qa_type breakdown."""
    per_row = []
    for i, (row, pred) in enumerate(zip(rows, preds)):
        gold = row["output"]
        gold_refs, pred_refs = extract_refs(gold), extract_refs(pred)
        hits = len(gold_refs & pred_refs)
        per_row.append({
            "qa_type": metas[i]["qa_type"] if metas else "all",
            "em": exact_match(pred, gold),
            "f1": token_f1(pred, gold),
            "cite_p": hits / len(pred_refs) if pred_refs else (1.0 if not gold_refs else 0.0),
            "cite_r": hits / len(gold_refs) if gold_refs else 1.0,
            "cite_all": float(gold_refs <= pred_refs),
            # cite_all is a subset test, so an answer that names the right old
            # section and invents a wrong new one still passes it. Requiring
            # the sets to match catches that.
            "cite_exact": float(gold_refs == pred_refs),
            "has_refs": bool(gold_refs),
        })
    return per_row


def summarise(per_row, label):
    def agg(rows_, key):
        vals = [r[key] for r in rows_]
        return 100 * sum(vals) / len(vals) if vals else float("nan")

    # Some question types (transition, scope) have no statutory citation in the
    # gold answer at all, so the citation metrics are simply not defined there
    # and are reported as None rather than as a misleading zero.
    cited = [r for r in per_row if r["has_refs"]]
    if cited:
        cp, cr = agg(cited, "cite_p"), agg(cited, "cite_r")
        cf1 = 2 * cp * cr / (cp + cr) if (cp + cr) else 0.0
        all_cites = agg(cited, "cite_all")
        exact_cites = agg(cited, "cite_exact")
    else:
        cf1 = all_cites = exact_cites = None
    return {
        "set": label, "n": len(per_row),
        "exact_match": agg(per_row, "em"),
        "f1": agg(per_row, "f1"),
        "citation_f1": cf1,
        "all_citations_present": all_cites,
        "citation_exact": exact_cites,
        "n_with_citations": len(cited),
    }


def pct(v):
    """Format a metric that may be undefined for this slice."""
    return "     n/a" if v is None else f"{v:>7.1f}%"

In [ ]:
confusion = [json.loads(l) for l in
             open(os.path.join(DATA, "confusion_test_set.jsonl"), encoding="utf-8")
             if l.strip()]

def answer_all(label, **kw):
    preds = []
    for i, e in enumerate(confusion):
        preds.append(bot.ask(e["instruction"], **kw).text)
        print(f"\r{label}: {i + 1}/{len(confusion)}", end="")
    print()
    return preds

# Phase 2.5 behaviour: one similarity passage, no concordance lookup.
saved = bot.counterparts
bot.counterparts = {}
single = answer_all("single passage", k=1, max_chunks=1)
bot.counterparts = saved

full = answer_all("bot context", k=3, max_chunks=4)

In [ ]:
metas = [{"qa_type": e["mapping_type"]} for e in confusion]
runs = {"single passage (Phase 2.5)": score(confusion, single, metas),
        "bot: + counterparts": score(confusion, full, metas)}

print(f"{'context assembly':<30}{'n':>5}{'F1':>9}{'citeF1':>9}{'allCites':>10}")
print("-" * 63)
summary = {}
for label, rows in runs.items():
    s = summarise(rows, label)
    summary[label] = s
    print(f"{label:<30}{s['n']:>5}{pct(s['f1'])}{pct(s['citation_f1'])}"
          f"{pct(s['all_citations_present'])}")

print(f"\n{'kind':<30}{'single':>12}{'bot':>12}   (allCites)")
print("-" * 56)
by_kind = {}
for kind in sorted({m["qa_type"] for m in metas}):
    vals = []
    for label, rows in runs.items():
        sub = [r for r in rows if r["qa_type"] == kind]
        vals.append(summarise(sub, kind))
    by_kind[kind] = {"single": vals[0], "bot": vals[1]}
    print(f"{kind:<30}{pct(vals[0]['all_citations_present']):>12}"
          f"{pct(vals[1]['all_citations_present']):>12}")

In [ ]:
# Side by side, so the difference is readable rather than only scored.
for kind in ("collision", "merged", "split", "removed"):
    i = next((i for i, e in enumerate(confusion)
              if e["mapping_type"] == kind), None)
    if i is None:
        continue
    e = confusion[i]
    want = extract_refs(e["output"])
    print("=" * 78)
    print(f"[{kind}] {e['instruction']}")
    print("  gold  :", textwrap.shorten(e["output"], 180, placeholder=" ..."))
    for label, preds in (("single", single), ("bot   ", full)):
        ok = want <= extract_refs(preds[i])
        print(f"  {label}: [{'OK ' if ok else 'MISS'}] "
              + textwrap.shorten(preds[i], 170, placeholder=" ..."))

---

## Saving the results

In [ ]:
out = {
    "run": "phase3 bot",
    "model_dir": MODEL_DIR,
    "confusion_overall": {k: v for k, v in summary.items()},
    "confusion_by_kind": by_kind,
    "demo": [{"question": q, "answer": bot.ask(q).text,
              "citations": bot.ask(q).citations} for q in QUESTIONS[:4]],
}
dest = os.path.join(DRIVE_DIR, "phase3_bot_results.json")
with open(dest, "w", encoding="utf-8") as fh:
    json.dump(out, fh, indent=2)
print("written to", dest)

---

## Try your own questions

Run the cell below and type anything. Blank line to stop.

In [ ]:
while True:
    q = input("\nQuestion (blank to stop): ").strip()
    if not q:
        break
    a = bot.ask(q)
    print()
    print(textwrap.fill(a.text, 76))
    for c in a.citations:
        print(f"  source: {c['source']}  {c['source_url']}")
    print(f"\n{DISCLAIMER}")

---

## What is left

**Phase 4** puts the confusion set to a general-purpose LLM — ChatGPT, Claude or
Gemini — and reports three columns side by side: that model, this bot without
retrieval, and this bot with it. That is the comparison the whole project was
built to make, and the data for it is already fixed and held out.

Then the paper and the video, for which the numbers live in `RESULTS.md`.